In [5]:
from pathlib import Path
from collections import defaultdict
import csv
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "/Users/gupta/Documents/DIS-IND/data/t2d/processed"

# Rows processed at once
CHUNK_SIZE = 100_000


# ============================================================
# CLEAN VALUES
# ============================================================

def normalize_value(value):
    """
    Normalize a value before distinct-value and cluster analysis.

    Example:
        '$38\\t152\\t000'
    becomes
        '$38 152 000'

    This avoids tabs/newlines creating unnecessary differences.
    """

    if value is None:
        return ""

    value = str(value)

    value = value.replace("\t", " ")
    value = value.replace("\r", " ")
    value = value.replace("\n", " ")

    # Collapse repeated whitespace
    value = " ".join(value.split())

    return value.strip()


# ============================================================
# MAIN ANALYZER
# ============================================================

def analyze_dataset(
    dataset_path,
    chunk_size=100_000
):

    dataset_path = Path(dataset_path)


    # ========================================================
    # FIND FILES
    # ========================================================

    csv_files = sorted(
        dataset_path.glob("*.csv")
    )

    tbl_files = sorted(
        dataset_path.glob("*.tbl")
    )


    if csv_files and tbl_files:

        raise ValueError(
            "Dataset contains both CSV and TBL files.\n"
            "Expected only one file format per dataset."
        )


    if csv_files:

        files = csv_files
        file_type = "csv"

    elif tbl_files:

        files = tbl_files
        file_type = "tbl"

    else:

        raise FileNotFoundError(
            f"No .csv or .tbl files found in:\n"
            f"{dataset_path}"
        )


    # ========================================================
    # GLOBAL STATISTICS
    # ========================================================

    number_of_tables = len(files)

    total_rows = 0

    max_rows_per_table = 0

    total_attributes = 0

    # Distinct counts of all attributes
    all_attribute_distinct_counts = []


    # --------------------------------------------------------
    # Unique values over COMPLETE dataset
    # --------------------------------------------------------

    all_dataset_values = set()


    # --------------------------------------------------------
    # For clusters:
    #
    # value -> attributes containing that value
    #
    # Example:
    #
    # 1 -> {A, B, C}
    # 2 -> {A, B}
    # 3 -> {A, B, D}
    # 4 -> {A, B}
    #
    # unique attribute combinations:
    #
    # {A,B,C}
    # {A,B}
    # {A,B,D}
    #
    # clusters = 3
    # --------------------------------------------------------

    value_to_attributes = defaultdict(set)


    # Per-table reports
    table_reports = []


    # ========================================================
    # DATASET SIZE
    # ========================================================

    total_size_bytes = sum(
        file_path.stat().st_size
        for file_path in files
    )

    total_size_mb = (
        total_size_bytes
        / (1024 * 1024)
    )


    # ========================================================
    # PROCESS EVERY TABLE
    # ========================================================

    for file_path in files:

        print()
        print("=" * 80)
        print(f"Analyzing: {file_path.name}")
        print("=" * 80)


        # ----------------------------------------------------
        # Detect separator
        # ----------------------------------------------------

        separator = "," if file_type == "csv" else "|"

        print(
            f"Detected separator : {repr(separator)}"
        )


        # ----------------------------------------------------
        # Table size
        # ----------------------------------------------------

        table_size_mb = (
            file_path.stat().st_size
            / (1024 * 1024)
        )


        table_rows = 0


        # ====================================================
        # DETERMINE COLUMNS
        # ====================================================

        if file_type == "csv":

            header = pd.read_csv(
                file_path,
                sep=separator,
                nrows=0,
                skip_blank_lines=True
            )

            columns = [
                str(column).strip()
                for column in header.columns
            ]


        else:

            # TBL normally has no header

            sample = pd.read_csv(
                file_path,

                sep=separator,

                header=None,

                nrows=1,

                dtype=str,

                engine="python",

                quoting=csv.QUOTE_NONE,

                skip_blank_lines=True
            )


            # TBL files often end rows with:
            #
            # value1|value2|value3|
            #
            # which creates an empty extra column.

            if (
                len(sample.columns) > 0
                and
                sample.iloc[:, -1].isna().all()
            ):

                number_of_columns = (
                    len(sample.columns) - 1
                )

            else:

                number_of_columns = len(
                    sample.columns
                )


            columns = [
                f"column_{i + 1}"
                for i in range(number_of_columns)
            ]


        number_of_attributes = len(
            columns
        )


        total_attributes += (
            number_of_attributes
        )


        # ====================================================
        # DISTINCT VALUES PER ATTRIBUTE
        # ====================================================

        distinct_values = {

            column: set()

            for column in columns

        }


        # ====================================================
        # CREATE CHUNK READER
        # ====================================================

        if file_type == "csv":

            reader = pd.read_csv(
                file_path,
                sep=separator,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False,
                skip_blank_lines=True,
                on_bad_lines="error"
            )


        else:

            reader = pd.read_csv(
                file_path,

                sep=separator,

                header=None,

                chunksize=chunk_size,

                dtype=str,

                keep_default_na=False,

                skip_blank_lines=True,

            )


        # ====================================================
        # PROCESS CHUNKS
        # ====================================================

        for chunk in reader:


            # ------------------------------------------------
            # Remove extra trailing TBL column
            # ------------------------------------------------

            if (
                file_type == "tbl"
                and
                len(chunk.columns) > len(columns)
            ):

                chunk = chunk.iloc[
                    :,
                    :len(columns)
                ]


            # ------------------------------------------------
            # Ensure expected number of columns
            # ------------------------------------------------

            if len(chunk.columns) > len(columns):

                chunk = chunk.iloc[
                    :,
                    :len(columns)
                ]


            if len(chunk.columns) < len(columns):

                print(
                    "Warning: malformed chunk encountered. "
                    "Some columns are missing."
                )

                # Add missing columns
                while (
                    len(chunk.columns)
                    < len(columns)
                ):

                    chunk[
                        f"_missing_{len(chunk.columns)}"
                    ] = ""


            chunk.columns = columns


            # ------------------------------------------------
            # REMOVE COMPLETELY EMPTY ROWS
            # ------------------------------------------------

            # Your example contains many blank rows at the end.
            # They should not count as tuples.

            non_empty_mask = (
                chunk
                .astype(str)
                .apply(
                    lambda row:
                    any(
                        normalize_value(v) != ""
                        for v in row
                    ),
                    axis=1
                )
            )

            chunk = chunk[
                non_empty_mask
            ]


            # ------------------------------------------------
            # Count rows
            # ------------------------------------------------

            table_rows += len(chunk)


            # =================================================
            # PROCESS EACH ATTRIBUTE
            # =================================================

            for column in columns:


                # ---------------------------------------------
                # Normalize values
                # ---------------------------------------------

                values = chunk[column].map(
                    normalize_value
                )


                # ---------------------------------------------
                # REMOVE EMPTY VALUES
                #
                # Empty strings are NOT considered actual
                # distinct data values here.
                # ---------------------------------------------

                values = values[
                    values != ""
                ]


                unique_values = values.unique()


                # ---------------------------------------------
                # DISTINCT VALUES FOR ATTRIBUTE
                # ---------------------------------------------

                distinct_values[
                    column
                ].update(
                    unique_values
                )


                # ---------------------------------------------
                # DISTINCT VALUES IN WHOLE DATASET
                # ---------------------------------------------

                all_dataset_values.update(
                    unique_values
                )


                # ---------------------------------------------
                # QUALIFIED ATTRIBUTE NAME
                #
                # Example:
                #
                # movies.Film
                # orders.Film
                # ---------------------------------------------

                qualified_attribute = (
                    f"{file_path.stem}.{column}"
                )


                # ---------------------------------------------
                # CLUSTER INFORMATION
                # ---------------------------------------------

                for value in unique_values:

                    value_to_attributes[
                        value
                    ].add(
                        qualified_attribute
                    )


        # ====================================================
        # TABLE STATISTICS
        # ====================================================

        total_rows += table_rows


        max_rows_per_table = max(
            max_rows_per_table,
            table_rows
        )


        # ----------------------------------------------------
        # Distinct statistics
        # ----------------------------------------------------

        table_distinct_counts = []


        for column in columns:

            distinct_count = len(
                distinct_values[column]
            )


            table_distinct_counts.append(
                distinct_count
            )


            all_attribute_distinct_counts.append(
                distinct_count
            )


        # ----------------------------------------------------
        # Max / average distinct for this table
        # ----------------------------------------------------

        if table_distinct_counts:

            table_max_distinct = max(
                table_distinct_counts
            )


            table_average_distinct = (
                sum(table_distinct_counts)
                /
                len(table_distinct_counts)
            )

        else:

            table_max_distinct = 0

            table_average_distinct = 0


        # ----------------------------------------------------
        # Store table result
        # ----------------------------------------------------

        table_reports.append({

            "table":
                file_path.name,

            "size_mb":
                table_size_mb,

            "rows":
                table_rows,

            "attributes":
                number_of_attributes,

            "max_distinct":
                table_max_distinct,

            "average_distinct":
                table_average_distinct

        })


        print(
            f"Rows               : "
            f"{table_rows:,}"
        )

        print(
            f"Attributes         : "
            f"{number_of_attributes:,}"
        )

        print(
            f"Size               : "
            f"{table_size_mb:,.2f} MB"
        )

        print(
            f"Max distinct       : "
            f"{table_max_distinct:,}"
        )

        print(
            f"Average distinct   : "
            f"{table_average_distinct:,.2f}"
        )


        # Free per-table distinct sets
        del distinct_values


    # ========================================================
    # DATASET-WIDE DISTINCT STATISTICS
    # ========================================================

    if all_attribute_distinct_counts:

        max_distinct_values_per_attribute = max(
            all_attribute_distinct_counts
        )


        average_distinct_values_per_attribute = (
            sum(all_attribute_distinct_counts)
            /
            len(all_attribute_distinct_counts)
        )

    else:

        max_distinct_values_per_attribute = 0

        average_distinct_values_per_attribute = 0


    # ========================================================
    # TOTAL DISTINCT VALUES IN DATASET
    # ========================================================

    total_distinct_values_dataset = len(
        all_dataset_values
    )


    # ========================================================
    # COUNT CLUSTERS
    # ========================================================
    #
    # We DO NOT print clusters.
    #
    # We only count unique attribute-occurrence combinations.
    #
    # Example:
    #
    # 1 -> {A,B,C}
    # 2 -> {A,B}
    # 3 -> {A,B,D}
    # 4 -> {A,B}
    #
    # produces:
    #
    # {A,B,C}
    # {A,B}
    # {A,B,D}
    #
    # Total clusters = 3
    # ========================================================

    unique_attribute_sets = set()


    for attributes in value_to_attributes.values():

        unique_attribute_sets.add(
            frozenset(attributes)
        )


    number_of_clusters = len(
        unique_attribute_sets
    )


    # ========================================================
    # FINAL DATASET SUMMARY
    # ========================================================

    print()
    print()
    print("=" * 90)
    print("FINAL DATASET SUMMARY")
    print("=" * 90)


    print(
        f"Number of tables                       : "
        f"{number_of_tables:,}"
    )


    print(
        f"Dataset size (MB)                      : "
        f"{total_size_mb:,.2f}"
    )


    print(
        f"Total rows                             : "
        f"{total_rows:,}"
    )


    print(
        f"Max # rows / tuples per table          : "
        f"{max_rows_per_table:,}"
    )


    print(
        f"Total # attributes                     : "
        f"{total_attributes:,}"
    )


    print(
        f"Max # distinct values per attribute    : "
        f"{max_distinct_values_per_attribute:,}"
    )


    print(
        f"Average # distinct values per attribute: "
        f"{average_distinct_values_per_attribute:,.2f}"
    )


    print(
        f"Total distinct values in dataset       : "
        f"{total_distinct_values_dataset:,}"
    )


    print(
        f"Total # clusters                       : "
        f"{number_of_clusters:,}"
    )


    # ========================================================
    # TABLE SUMMARY
    # ========================================================

    print()
    print("=" * 90)
    print("TABLE SUMMARY")
    print("=" * 90)


    for table in table_reports:

        print()

        print(
            f"Table             : "
            f"{table['table']}"
        )

        print(
            f"Size (MB)         : "
            f"{table['size_mb']:,.2f}"
        )

        print(
            f"Rows              : "
            f"{table['rows']:,}"
        )

        print(
            f"Attributes        : "
            f"{table['attributes']:,}"
        )

        print(
            f"Max distinct      : "
            f"{table['max_distinct']:,}"
        )

        print(
            f"Average distinct  : "
            f"{table['average_distinct']:,.2f}"
        )


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {

        "number_of_tables":
            number_of_tables,

        "dataset_size_mb":
            total_size_mb,

        "total_rows":
            total_rows,

        "max_rows_per_table":
            max_rows_per_table,

        "total_attributes":
            total_attributes,

        "max_distinct_values_per_attribute":
            max_distinct_values_per_attribute,

        "average_distinct_values_per_attribute":
            average_distinct_values_per_attribute,

        "total_distinct_values_dataset":
            total_distinct_values_dataset,

        "number_of_clusters":
            number_of_clusters,

        "tables":
            table_reports

    }


# ============================================================
# RUN
# ============================================================

report = analyze_dataset(
    DATASET_PATH,
    chunk_size=CHUNK_SIZE
)


Analyzing: 10151359_0_8168779773862259178.csv
Detected separator : ','
Rows               : 151
Attributes         : 3
Size               : 0.01 MB
Max distinct       : 151
Average distinct   : 100.00

Analyzing: 10579449_0_1681126353774891032.csv
Detected separator : ','
Rows               : 21
Attributes         : 3
Size               : 0.00 MB
Max distinct       : 21
Average distinct   : 19.00

Analyzing: 10630177_0_4831842476649004753.csv
Detected separator : ','
Rows               : 200
Attributes         : 7
Size               : 0.01 MB
Max distinct       : 200
Average distinct   : 97.14

Analyzing: 11278409_0_3742771475298785475.csv
Detected separator : ','
Rows               : 206
Attributes         : 5
Size               : 0.01 MB
Max distinct       : 191
Average distinct   : 83.40

Analyzing: 1146722_1_7558140036342906956.csv
Detected separator : ','
Rows               : 215
Attributes         : 7
Size               : 0.01 MB
Max distinct       : 215
Average distinct   : 146